# CM-Expert-LLM: Data Ingestion Tutorial

This notebook demonstrates how to ingest condensed matter physics literature into the CM-Expert-LLM system for training and RAG.

## Overview

The data ingestion pipeline:
1. Reads raw text/markdown/PDF files from `data/raw/`
2. Chunks text into manageable pieces (800 words with 120 word overlap)
3. Extracts metadata (source, type, character count)
4. Outputs JSONL format for training and retrieval

## Setup

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd() / 'src'))

from cmp_expert.data.pipeline import process_directory, chunk_text, extract_metadata

print("✓ Imports successful")

## Step 1: Prepare Raw Data

Place your condensed matter physics documents in `data/raw/`. Supported formats:
- `.txt` - Plain text files
- `.md` - Markdown files
- `.tex` - LaTeX source files

Example sources:
- arXiv papers (cond-mat)
- Textbook chapters
- Lecture notes
- Research summaries

In [ ]:
# Create sample data for demonstration
raw_dir = Path('data/raw')
raw_dir.mkdir(exist_ok=True)

sample_text = """
# Introduction to Superconductivity

## The Meissner Effect

The Meissner effect is the expulsion of a magnetic field from a superconductor during its transition to the superconducting state. This phenomenon was discovered by Walther Meissner and Robert Ochsenfeld in 1933.

When a material transitions into the superconducting state, it expels magnetic flux from its interior. This is not simply perfect diamagnetism, but a fundamental property of the superconducting state itself.

## BCS Theory

The BCS theory, proposed by Bardeen, Cooper, and Schrieffer in 1957, explains conventional superconductivity as a result of Cooper pair formation. Electrons form pairs through phonon-mediated attraction, leading to a macroscopic quantum state.

The energy gap Δ in the quasiparticle spectrum is given by:
Δ = ℏω_D exp(-1/N(0)V)

where ω_D is the Debye frequency, N(0) is the density of states at the Fermi level, and V is the pairing interaction strength.

## Critical Temperature

The critical temperature Tc marks the transition to the superconducting state. For conventional superconductors, Tc is typically below 30K. High-temperature superconductors (cuprates) can have Tc above 100K.
"""

sample_file = raw_dir / 'superconductivity_intro.md'
sample_file.write_text(sample_text)
print(f"✓ Created sample file: {sample_file}")

## Step 2: Run Ingestion Pipeline

In [ ]:
# Run the ingestion pipeline
output_file = Path('data/processed/sft.jsonl')
output_file.parent.mkdir(exist_ok=True)

process_directory(
    input_dir=raw_dir,
    output_file=output_file,
    chunk_size=800,
    overlap=120
)

print(f"✓ Ingestion complete: {output_file}")

## Step 3: Inspect Results

In [ ]:
import json

# Read and display chunks
with open(output_file, 'r') as f:
    chunks = [json.loads(line) for line in f]

print(f"Total chunks created: {len(chunks)}\n")

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(f"  ID: {chunk['id']}")
    print(f"  Text: {chunk['text'][:150]}...")
    print(f"  Metadata: {chunk['metadata']}\n")

## Step 4: Test Chunking Function

In [ ]:
# Test chunking with different parameters
test_text = "This is a test. " * 100  # 400 words

print("Chunking test:")
print(f"Original length: {len(test_text.split())} words\n")

for chunk_size in [200, 100, 50]:
    chunks = chunk_text(test_text, chunk_size=chunk_size, overlap=20)
    print(f"Chunk size {chunk_size}: {len(chunks)} chunks")

print("\n✓ Chunking test complete")

## Next Steps

1. Add more documents to `data/raw/`
2. Re-run the ingestion pipeline
3. Proceed to training with `train_lora.py`
4. Test retrieval with the RAG API

## See Also

- `02_training.ipynb` - Fine-tuning with LoRA
- `03_evaluation.ipynb` - Benchmark evaluation
- `04_rag_serving.ipynb` - RAG API deployment